# Figure 3: Risk percentile distributions for Top 10% successes and misses

This revised notebook creates Figure 3 from the 14_v5 strict first-occurrence evaluation results. It does not search for input files automatically. Place `14_v5_case_results_same_extratrees_leakage_safe.csv` in the current working directory, or edit `CASE_RESULTS_PATH` below.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ============================================================
# Figure 3: Risk percentile distributions for Top 10% successes
# and misses among strict first-occurrence-like events
# under the final v7 baseline Extra Trees model.
# ============================================================

candidate_paths = [
    Path(
        "/content/drive/MyDrive/avian_influenza_project/processed/"
        "model_outputs_riskmap_eval/"
        "14_v7_baseline_top10_success_miss_cases.csv"
    ),
    Path("14_v7_baseline_top10_success_miss_cases.csv"),
]

CASE_RESULTS_PATH = next(
    (path for path in candidate_paths if path.exists()),
    None,
)

if CASE_RESULTS_PATH is None:
    raise FileNotFoundError(
        "14_v7_baseline_top10_success_miss_cases.csv was not found.\n"
        "Checked:\n"
        + "\n".join(str(path) for path in candidate_paths)
    )

OUT_DIR = CASE_RESULTS_PATH.parent
PNG_PATH = (
    OUT_DIR
    / "figure3_risk_percentile_top10_success_miss_14v7.png"
)
PDF_PATH = (
    OUT_DIR
    / "figure3_risk_percentile_top10_success_miss_14v7.pdf"
)

df = pd.read_csv(CASE_RESULTS_PATH)

required_cols = {
    "risk_percentile",
    "top10_group",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}\n"
        f"Available columns: {df.columns.tolist()}"
    )

plot_df = df.dropna(
    subset=["risk_percentile", "top10_group"]
).copy()

plot_df["risk_percentile"] = pd.to_numeric(
    plot_df["risk_percentile"],
    errors="coerce",
)
plot_df = plot_df.dropna(
    subset=["risk_percentile"]
).copy()

expected_counts = {
    "Top 10 miss": 38,
    "Top 10 success": 11,
}

observed_counts = (
    plot_df["top10_group"]
    .value_counts()
    .reindex(expected_counts.keys(), fill_value=0)
    .to_dict()
)

print("Using:", CASE_RESULTS_PATH)
print("Observed counts:", observed_counts)

if observed_counts != expected_counts:
    raise ValueError(
        "The category counts do not match the final v7 results.\n"
        f"Expected: {expected_counts}\n"
        f"Observed: {observed_counts}"
    )

if len(plot_df) != 49:
    raise ValueError(
        f"Expected 49 events, but found {len(plot_df)}."
    )

summary = (
    plot_df.groupby("top10_group")["risk_percentile"]
    .agg(
        events="count",
        mean="mean",
        median="median",
        minimum="min",
        maximum="max",
    )
    .reindex(["Top 10 miss", "Top 10 success"])
)

print("\nRisk-percentile summary:")
display(summary)

order = ["Top 10 miss", "Top 10 success"]
labels = ["Top 10% miss", "Top 10% success"]

data = [
    plot_df.loc[
        plot_df["top10_group"] == group,
        "risk_percentile",
    ].to_numpy()
    for group in order
]

fig, ax = plt.subplots(figsize=(6.6, 4.8))

bp = ax.boxplot(
    data,
    labels=labels,
    showmeans=True,
    patch_artist=True,
    meanprops={
        "marker": "^",
        "markerfacecolor": "black",
        "markeredgecolor": "black",
        "markersize": 6,
    },
    medianprops={
        "color": "black",
        "linewidth": 1.5,
    },
    boxprops={
        "edgecolor": "black",
        "linewidth": 1.2,
    },
    whiskerprops={
        "color": "black",
        "linewidth": 1.0,
    },
    capprops={
        "color": "black",
        "linewidth": 1.0,
    },
    flierprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 5,
        "linestyle": "none",
    },
)

for patch, color in zip(
    bp["boxes"],
    ["lightgray", "lightblue"],
):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

rng = np.random.default_rng(42)
for x_position, group in enumerate(order, start=1):
    values = plot_df.loc[
        plot_df["top10_group"] == group,
        "risk_percentile",
    ].to_numpy()
    jitter = rng.normal(
        loc=x_position,
        scale=0.035,
        size=len(values),
    )
    ax.scatter(
        jitter,
        values,
        s=28,
        alpha=0.70,
        edgecolors="black",
        linewidths=0.4,
        zorder=3,
    )

ax.axhline(
    0.9,
    linestyle="--",
    linewidth=1.0,
    color="black",
    alpha=0.7,
)
ax.text(
    0.03,
    0.905,
    "Top 10% threshold",
    transform=ax.get_yaxis_transform(),
    va="bottom",
    ha="left",
    fontsize=9,
)

ax.set_ylabel("Event-grid risk percentile")
ax.set_xlabel("Case group")
ax.set_ylim(0, 1.05)
ax.grid(
    axis="y",
    linewidth=0.4,
    alpha=0.5,
)

plt.tight_layout()

plt.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    PDF_PATH,
    bbox_inches="tight",
)
plt.show()

print("Saved:", PNG_PATH)
print("Saved:", PDF_PATH)
